# Smart City-project van Nova Stad

In [33]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pyspark.sql.functions import col, lit, udf, pandas_udf, PandasUDFType
from pyspark.sql.types import *
from pyspark.ml.feature import StringIndexer, VectorAssembler, StandardScaler
from pyspark.ml import Pipeline
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator

from pyspark.ml.tuning import ParamGridBuilder, CrossValidator
from pyspark.ml.regression import RandomForestRegressor
from pyspark.ml.regression import GBTRegressor
from pyspark.ml.regression import DecisionTreeRegressor
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, VectorAssembler, StandardScaler, OneHotEncoder, Imputer
from pyspark.ml.classification import DecisionTreeClassifier, RandomForestClassifier, GBTClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator

import os
import json
from PIL import Image

ModuleNotFoundError: No module named 'pyspark'

# Data inladen

In [ ]:
img_path = "bdd/images/train"
label_path = "bdd/json_labels/train"

# haal image namen zonder .jpg
image_ids = set([f.replace(".jpg", "") for f in os.listdir(img_path) if f.endswith(".jpg")])

# labels ophalen
labels = [f for f in os.listdir(label_path) if f.endswith(".json")]

# check matches
matched = [f for f in labels if f.replace(".json", "") in image_ids]

print("Aantal images", len(image_ids))
print("Aantal labels", len(labels))
print("Matches", len(matched))

Aantal images 2476
Aantal labels 2476
Matches 2476


### label verwijderen

In dit code gedeelte checken we matching labels en images. Niet gekoppelde items worden verwijderd.

In [37]:
matched_set = set([f.replace(".json", "") for f in matched])

In [38]:
for f in labels:
    base = f.replace(".json", "")
    
    if base not in matched_set:
        os.remove(os.path.join(label_path, f))

print("Alleen matched labels behouden ")

Alleen matched labels behouden 


In [39]:
remaining_labels = [f for f in os.listdir(label_path) if f.endswith(".json")]

print("Overgebleven labels", len(remaining_labels))

Overgebleven labels 2476


### Images verwijderen

In [40]:
image_files = [f for f in os.listdir(img_path) if f.endswith(".jpg")]
label_ids = set(os.path.splitext(f)[0] for f in os.listdir(label_path) if f.endswith(".json"))

delete_images = [f for f in image_files if os.path.splitext(f)[0] not in label_ids]


In [41]:
for f in delete_images:
    os.remove(os.path.join(img_path, f))

print("Images opgeschoond")

Images opgeschoond


In [ ]:
remaining_images = [f for f in os.listdir(img_path) if f.endswith(".jpg")]
remaining_labels = [f for f in os.listdir(label_path) if f.endswith(".json")]

print("Images", len(remaining_images))
print("Labels", len(remaining_labels))

Images: 2476
Labels: 2476


## Verdeel images en labels naar val map

In [43]:
import os
import shutil
import random

img_path = "bdd/images/train"
val_img = "bdd/images/val"
label_path = "bdd/json_labels/train"
val_label = "bdd/json_labels/val"

os.makedirs(val_img, exist_ok=True)
os.makedirs(val_label, exist_ok=True)

#  Verwijder alle val images
#for f in os.listdir(val_img):
    #if f.endswith(".jpg"):
        #os.remove(os.path.join(val_img, f))

# Verwijder alle val labels
#for f in os.listdir(val_label):
    #if f.endswith(".json"):
        #os.remove(os.path.join(val_label, f))

# Kies alleen train images die ook een label hebben
image_files = [f for f in os.listdir(img_path) if f.endswith(".jpg")]
label_ids = {os.path.splitext(f)[0] for f in os.listdir(label_path) if f.endswith(".json")}


matched_images = [f for f in image_files if os.path.splitext(f)[0] in label_ids]

print("Beschikbare matched train images", len(matched_images))

# Selecteer 500 images voor val
#random.seed(42)
#val_selection = random.sample(matched_images, 500)

# Verplaats image + label naar val
#for img_file in val_selection:
    #base = os.path.splitext(img_file)[0]
    #label_file = base + ".json"

    #shutil.move(os.path.join(img_path, img_file), os.path.join(val_img, img_file))
    #shutil.move(os.path.join(label_path, label_file), os.path.join(val_label, label_file))

#print("Klaar")

Beschikbare matched train images 2476


### Check aantal files

In [44]:
def count_files(path, ext):
    return len([f for f in os.listdir(path) if f.endswith(ext)])

print("Train images:", count_files(img_path, ".jpg"))
print("Train labels:", count_files(label_path, ".json"))
print("Val images:", count_files(val_img, ".jpg"))
print("Val labels:", count_files(val_label, ".json"))

Train images: 2476
Train labels: 2476
Val images: 500
Val labels: 500


## Data pipeline

In [18]:
df_traffic = spark.read.csv("/FileStore/tables/metro_interstate_traffic_volume_train.csv", header=True, inferSchema=True)

NameError: name 'spark' is not defined

In [0]:
df = spark.read.csv(
    "/Volumes/main/default/datasets/metro_interstate_traffic_volume_train.csv",
    header=True,
    inferSchema=True
)

display(df.head(5))

In [0]:
# Imports
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, trim, regexp_replace, to_timestamp, year, month, dayofmonth, hour, dayofweek
from pyspark.sql.types import StringType

spark = SparkSession.builder.getOrCreate()

# Pad
train_path = "dbfs:/Workspace/Repos/20076436@student.hhs.nl/MLops-City/Metro_Interstate_Traffic_Volume_train.csv"
test_path  = "dbfs:/Workspace/Repos/20076436@student.hhs.nl/MLops-City/Metro_Interstate_Traffic_Volume_test.csv"
# functie voor schoonmaken
def clean_df(df):

    # 1. duplicaten verwijderen
    df = df.dropDuplicates()

    # 2. datum omzetten
    df = df.withColumn("date_time", to_timestamp(col("date_time"), "yyyy-MM-dd HH:mm:ss"))

    # 3. string kolommen opschonen
    string_cols = [f.name for f in df.schema.fields if isinstance(f.dataType, StringType)]
    for c in string_cols:
        df = df.withColumn(c, trim(col(c)))
        df = df.withColumn(c, regexp_replace(col(c), r"\s+", " "))

    # 4. numerieke kolommen bepalen
    numeric_types = ["int", "bigint", "double", "float", "long", "short", "decimal"]
    numeric_cols = [f.name for f in df.schema.fields if f.dataType.simpleString() in numeric_types]

    # 5. missende numerieke waardes vullen (mediaan)
    for c in numeric_cols:
        median_value = df.approxQuantile(c, [0.5], 0.01)[0]
        df = df.fillna({c: median_value})

    # 6. missende string waardes vullen
    for c in string_cols:
        df = df.fillna({c: "Unknown"})

    # 7. datum features maken
    df = (
        df
        .withColumn("year", year(col("date_time")))
        .withColumn("month", month(col("date_time")))
        .withColumn("day", dayofmonth(col("date_time")))
        .withColumn("hour", hour(col("date_time")))
        .withColumn("dayofweek", dayofweek(col("date_time")))
    )

    return df


# Alles in 1 lijn uitvoeren (dan wordt neiwue df een keer opgeslagen)
df_train = clean_df(
    spark.read.option("header", True).option("inferSchema", True).csv(train_path)
)

df_test = clean_df(
    spark.read.option("header", True).option("inferSchema", True).csv(test_path)
)

# Resultaat bekijken
display(df_train.limit(5))
display(df_test.limit(5))

# YOLO

### Bestand converten

ChatGPT 5.2, Prompt 2: JSON naar YOLO TXT converteren, https://chatgpt.com/share/6a0339c0-52ac-8390-bb0e-3daf97c80b68


In [45]:
# CLASS MAPPING

classes = {
    "car": 0,
    "bus": 1,
    "truck": 2,
    "person": 3,
    "traffic sign": 4,
    "traffic light": 5,
    "bike": 6,
    "motor": 7
}


# CONVERTER FUNCTIE

def convert_to_yolo(img_path, label_path, output_dir):

    os.makedirs(output_dir, exist_ok=True)

    for json_file in os.listdir(label_path):

        # alleen json files
        if not json_file.endswith(".json"):
            continue

        json_path = os.path.join(label_path, json_file)

        # json openen
        with open(json_path, "r") as f:
            data = json.load(f)

        # image naam ophalen
        image_name = data["name"] + ".jpg"

        image_path = os.path.join(img_path, image_name)

        # check image bestaat
        if not os.path.exists(image_path):
            print(f"Image niet gevonden: {image_name}")
            continue

        # image openen
        img = Image.open(image_path)

        img_width, img_height = img.size

        yolo_lines = []

        frames = data.get("frames", [])

        for frame in frames:

            objects = frame.get("objects", [])

            for obj in objects:

                category = obj.get("category")

                # alleen gewenste classes
                if category not in classes:
                    continue

                # alleen box2d objecten
                if "box2d" not in obj:
                    continue

                box = obj["box2d"]

                x1 = box["x1"]
                y1 = box["y1"]
                x2 = box["x2"]
                y2 = box["y2"]


                # YOLO CONVERSIE

                x_center = (x1 + x2) / 2
                y_center = (y1 + y2) / 2

                width = x2 - x1
                height = y2 - y1

                # NORMALISEREN

                x_center /= img_width
                y_center /= img_height

                width /= img_width
                height /= img_height

                class_id = classes[category]

                line = f"{class_id} {x_center} {y_center} {width} {height}"

                yolo_lines.append(line)

        # txt file opslaan
        txt_name = os.path.splitext(json_file)[0] + ".txt"

        txt_path = os.path.join(output_dir, txt_name)

        with open(txt_path, "w") as f:
            f.write("\n".join(yolo_lines))

        print(f"Geconverteerd {json_file}")

    print(f"Klaar {output_dir}")


# TRAIN MAP CONVERTEREN

convert_to_yolo(
    img_path="bdd/images/train",
    label_path="bdd/json_labels/train",
    output_dir="bdd/labels/train"
)

# VAL MAP CONVERTEREN

convert_to_yolo(
    img_path="bdd/images/val",
    label_path="bdd/json_labels/val",
    output_dir="bdd/labels/val"
)

Geconverteerd 16a4ac1c-bcfbdf08.json
Geconverteerd 65ebd4ae-9d467cca.json
Geconverteerd 69ed96b3-72164b3d.json
Geconverteerd 9bbbafe6-db7b36de.json
Geconverteerd 692fd125-b034ac2f.json
Geconverteerd 58208859-5316743a.json
Geconverteerd 0c154337-09dd2a21.json
Geconverteerd 89422aa3-66f39aa2.json
Geconverteerd af3d1d6b-3c9fe003.json
Geconverteerd 2ea46f67-bde41d89.json
Geconverteerd 291dee4f-d2d55c47.json
Geconverteerd 087f969b-b1584c3f.json
Geconverteerd 641ae97b-8bc8e3b9.json
Geconverteerd a48909f7-95506b41.json
Geconverteerd 5b8651a2-bfa5be55.json
Geconverteerd a9c77142-750fef71.json
Geconverteerd 2143d41a-e88cb702.json
Geconverteerd 933651d0-c44d04b1.json
Geconverteerd ae420565-b8ce7713.json
Geconverteerd 3f3a389d-00b0c924.json
Geconverteerd 9aead4d4-5bb03f38.json
Geconverteerd 314db341-693a7d4d.json
Geconverteerd 43655cb5-4969d939.json
Geconverteerd 89116b3a-29def645.json
Geconverteerd 0b9fa855-936aae7a.json
Geconverteerd 1b162169-b05c9ad0.json
Geconverteerd 5b7f03ae-45fe20ac.json
G

In [51]:
## check aantal files
print("Train images:", count_files(img_path, ".jpg"))
print("Train labels:", count_files(label_path, ".json"))
print("Val images:", count_files(val_img, ".jpg"))
print("Val labels:", count_files(val_label, ".json"))

Train images: 2476
Train labels: 2476
Val images: 500
Val labels: 500


### Model trainen

# test

In [53]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")
results = model.train( data="data.yaml", 
                        epochs=50, 
                        imgsz=640, 
                        batch=16, 
                        patience=10, 
                        
                        cache=True, 
                        workers=4, 
                        optimizer="AdamW", 
                        lr0=0.001,

                        hsv_h=0.015,
                        hsv_s=0.7,
                        hsv_v=0.4,

                        degrees=5,
                        translate=0.1,
                        scale=0.5,

                        fliplr=0.5,
                        amp=True,

                        project="traffic_yolo",

                        name="yolov8n_edge")

New https://pypi.org/project/ultralytics/8.4.49 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.23 🚀 Python-3.10.19 torch-2.10.0 CPU (Apple M4 Pro)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data.yaml, degrees=5, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolov8n_edge2, nbs=64, nms=False, opset=None, optim

/Users/thidaratchuram/venvs/tf310/lib/python3.10/site-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)
2026/05/12 20:31:33 INFO mlflow.tracking.fluent: Experiment with name 'traffic_yolo' does not exist. Creating a new experiment.
2026/05/12 20:31:36 INFO mlflow.tracking.fluent: Autologging successfully enabled for statsmodels.


MLflow: logging run_id(223b72c365b54709ba09de3ae22ae50e) to runs/mlflow
MLflow: view at http://127.0.0.1:5000 with 'mlflow server --backend-store-uri runs/mlflow'
MLflow: disable with 'yolo settings mlflow=False'
Image sizes 640 train, 640 val
Using 0 dataloader workers
Logging results to /Users/thidaratchuram/Documents/GitHub/MLops-City/runs/detect/traffic_yolo/yolov8n_edge2
Starting training for 50 epochs...

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       1/50         0G      1.691      1.745      1.093        332        640: 100% ━━━━━━━━━━━━ 155/155 3.2s/it 8:212.4ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 16/16 1.9s/it 30.4s1.9ss
                   all        500      10165      0.548      0.223      0.203      0.102

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       2/50         0G      1.632      1.313      1.084        590        640: 18% 

KeyboardInterrupt: 